# 内部机制 2：托管状态与页 arena——版本、COW、pin、预算

这是内部机制支线（系统向）的第 2 篇。支线第 1 篇跟着一份声明走完了签名编译管线：你写的数学如何被冻结、检查、签名成一份执行计划。这篇回答计划落地之后的问题——**状态（参数、版本、页）由谁拥有、按什么纪律变动？**

先说为什么这值得单独一篇。对做推理引擎的人来说，最贵的两样东西是显存和并发正确性。一旦训练数学进入引擎，问题立刻变尖锐：参数正在被更新，而解码正在读它。常见的答案要么加全局锁牺牲吞吐，要么允许读旧值但语义含混。PLLM 的回答是把所有可变状态收进**唯一所有者** `PageBundleArena` 的签名纪律里：

- **版本（Version）**：读取 pin 住一个不可变根，执行只认 pin 住的版本，不重新选；
- **COW**：写入发生在私有页上，已发布的版本原地不动；
- **发布（Publication）**：私有迁移原子可见，完成 receipt 结清之前旧页不回收。

整篇在纯 CPU 上运行：不构造 vLLM 引擎、不初始化 CUDA。我们直接驱动编译产物——`StateExecution` 和它拥有的 arena，构造方式与 `test/workloads/` 里的 CPU 证据完全一致。读完你应该能：解释 `StateRef`、lineage、版本的关系；用 `diagnostics().state_usage` 读出 arena 的所有权状态；说清「pin 没释放」「预算超了」分别在哪一步、以什么方式失败。

## 落地：从签名计划到一台 arena

支线第 1 篇的终点是一份 `PlanBundle`：状态的资源、布局、表示与额度已经被签名固定。`compile_backend` 把它落成运行时——对我们关心的状态部分，就是一台 `PageBundleArena`：按签名布局为每个字段分配固定容量的页池，池地址此后不再变化（捕获稳定性以此为前提）。

为什么需要一台「arena」而不是几个普通张量？看架构文档「运行时边界」一节的所有权表第一行：**参数、优化器字段、私有版本都归 PageBundleArena**；复用由精确 pin 与签发的完成 receipt 门控；布局与提供者额度是签名的；写入是字段级 COW。一句话：arena 是版本、代次与完成的唯一所有者，系统里不存在第二份状态真相——`PageBundleArena` 的 docstring 原话是 *Only this arena chooses logical versions*。

下面用一个真实的带状态负载来演示：Output LoRA。它声明了一个实例作用域的状态 `output_factors`，含 `a`（4×2）、`b`（2×7）两个 float32 字段和一个 SGD 优化器。实例作用域意味着整个引擎只有一条 lineage，所以你会看到准入上限被规划器定为 1——这不是我们手写的事实，而是从声明推出的。

代码做三件事：构造一份 CPU 后端事实（`BackendCompileFacts`，与测试证据同款）、编译出 `execution`、用 `admit(())` 准入实例状态。准入会建立 lineage、分配版本 0 的页并按声明初始化；`snapshot` 给你一份自持的 CPU 拷贝。我们预计看到：资源名 `output_factors`、每字段 8 页容量、792 字节预留，以及版本 0 的身份证 `StateRef(layout_id=0, lineage_id=1, version_id=0)`。

In [1]:
import torch
from dataclasses import replace

from pllm.authoring import BackendCompileFacts, TensorStepInput, WeightedCEBatch
from pllm.config import ExecutionOptions
from pllm.execution.runtime_facts import model_timeline
from pllm.execution.ptts.page_bundle_planning import PageBundleProviderFacts
from pllm.workloads.lora import OutputLoRAConfig, OutputLoRAWorkload
from pllm.contracts.backend_attachment import (
    AttachmentGraphMode, AttachmentHiddenResource, AttachmentTopology, BackendPlacementProbe)
from pllm.contracts.hidden_locator import HiddenLayerKind, HiddenPhase, HiddenSite
from pllm.contracts.packed_hidden import (
    PackedHiddenResourceBinding, PackedHiddenSnapshot, PackedHiddenSnapshotBatch)
from pllm.contracts.packed_rows import (
    PackedPhaseRow, PackedPhaseRowBatch, PackedSamplerRowBatch, PackedSamplerRowLink)
from pllm.contracts.request_identity import RequestIdentity
from pllm.contracts.semantic_graph import StateScope
from pllm.contracts.state_representation import StateDeviceKind
from pllm.contracts.tensor_layout import LayoutFieldId, TensorDType

torch.manual_seed(7)

# 与 test/workloads/ 中 CPU 证据完全相同的构造：一台假想的 CPU 后端，
# hidden_size=4、vocab=7，页池容量 8 页/字段
binding = PackedHiddenResourceBinding("-1", HiddenLayerKind.INDEX, HiddenSite.MODEL_OUTPUT, "model.norm")
probe = BackendPlacementProbe("test.sampler@1", "a" * 64, "b" * 64, "c" * 64,
    AttachmentTopology.IN_PROCESS, AttachmentGraphMode.EAGER, StateDeviceKind.CPU, 0, 1, 1,
    TensorDType.FLOAT32, 4, 8, (AttachmentHiddenResource(binding, HiddenPhase.DECODE),), ())
base_facts = BackendCompileFacts(probe, model_timeline(torch.device("cpu")),
    (PageBundleProviderFacts("test.page@1", 8),), 8, 65536, 7, vocab_size=7)

config = OutputLoRAConfig(rank=2, alpha=3.0, learning_rate=0.3, steps=3, seed=7)
workload = OutputLoRAWorkload(config)
facts = workload.specialize(base_facts, ExecutionOptions(training_rows=4))
execution = workload.compile_backend(facts).execution
arena = execution.arena

rep = execution.state_plan.representations[0]
print("状态资源:", rep.resource, "| 表示:", str(rep.representation))
print("页容量:", rep.page_capacity, "页/字段 | 预留字节:", rep.reserved_bytes,
      "| 准入 lineage 上限:", rep.max_admitted_lineages)
print("arena 设备:", arena.device, "| 计划签名:", execution.arena.plan.schedule_decision.signature[:16], "...")

initial, = execution.admit(())
original = execution.snapshot(initial)
print("\n准入得到版本:", initial, "| 字段形状:", [tuple(t.shape) for t in original])
print("a 字段前两行:\n", original[0][:2])
print("\n准入后的 state_usage:", execution.diagnostics().state_usage)

状态资源: output_factors | 表示: dense_page_arena
页容量: 8 页/字段 | 预留字节: 792 | 准入 lineage 上限: 1
arena 设备: cpu | 计划签名: deec4e56221d1022 ...

准入得到版本: StateRef(layout_id=0, lineage_id=1, version_id=0) | 字段形状: [(4, 2), (2, 7)]
a 字段前两行:
 tensor([[-0.0015,  0.0079],
        [ 0.0095, -0.0111]])

准入后的 state_usage: PageBundleUsage(reserved_bytes=792, logical_live_bytes=88, allocated_payload_bytes=88, reserved_padding_bytes=88, free_field_pages=14, live_field_pages=2, private_field_pages=0, retiring_field_pages=0, private_versions=0, retiring_versions=0, pinned_versions=0, pending_receipts=0, admission_capacity_failures=0)


## 一步训练里，状态是怎么动的

输出里的 `state_usage` 先混个眼熟，稍后逐字段拆解。先确认两点：`a` 是带 seed 的确定性初始化（你在自己机器上应看到一样的数）；准入上限 1 是规划器按实例作用域推出的。整个 arena 预留 792 字节，而版本 0 只占 88 字节载荷——`a` 的 32 加 `b` 的 56——其余是容量与对齐余量。

现在让训练真正发生。`execution.step(value)` 按计划走完所有锚点：准备、预测、训练、发布。盯三个东西：

1. **发布的版本对**：`publication.before_refs` → `after_refs`，版本 0 变版本 1，同一条 lineage；
2. **pin 见证**：我们在训练前 pin 住了版本 0。训练结束后用它读旧版本——应当与训练前**逐位一致**。这是「执行只认签名时 pin 住的版本」的直接证据：写入从头到尾发生在别处；
3. **trace 形态**：`begin`（带着 pin 住的 refs）→ `sampler_complete` → `publish`（携带 `after_refs` 与 `operation_id`）→ `settled`（pin 与 release 结清）。

你可能会问：为什么读不到「写了一半」的参数？因为发布是**原子换根**——`latest` 在一次切换中指向新根，源码里明确写着「可见性切换与签发 receipt 之间不允许任何张量工作或用户代码」。写一半的状态存在于私有页上，对世界不可见。

In [2]:
from pllm.execution.trace import TracePhase

# 在训练步之前 pin 住版本 0：这是对「执行只认 pin 住的版本」的直接见证
old_pin = arena.pin_exact((initial,))

hidden3 = torch.tensor([[0.2, -0.1, 0.4, 0.3], [-0.5, 0.7, 0.1, -0.2], [0.1, 0.3, -0.2, 0.8]])
batch = WeightedCEBatch(torch.cat((hidden3, torch.full((1, 4), float("nan")))),
    torch.linspace(-1, 1, 28).reshape(4, 7), torch.tensor([2, 4, 1, -100]),
    torch.tensor([1., 3., 0., 1.]))
rows = PackedPhaseRowBatch(1, 0, 3, tuple(PackedPhaseRow(i, i, 0,
    HiddenPhase.DECODE, RequestIdentity(str(i), i, 0)) for i in range(3)))
value = TensorStepInput(rows, PackedSamplerRowBatch(rows, tuple(PackedSamplerRowLink(i, i) for i in range(3))),
    PackedHiddenSnapshotBatch(rows, (PackedHiddenSnapshot(binding, hidden3),)), auxiliary=batch)

result = execution.step(value)
publication = result.publication
latest, = publication.after_refs
print("loss:", [round(float(x), 4) for x in result.loss])
print("发布: 版本", publication.before_refs[0].version_id, "-> 版本", latest.version_id,
      "| operation_id:", publication.operation_id)
trained = execution.snapshot(latest)
print("两个字段都被更新:", [not torch.equal(a, b) for a, b in zip(trained, original)])

# 见证一：pinned 的旧版本逐位不变——写发生在私有页上，不是在原位改
same = all(bool(torch.equal(a, b)) for a, b in zip(arena.snapshot(old_pin, initial), original))
print("版本 0 经 old_pin 读出，与训练前逐位一致:", same)
print("arena.latest 已指向:", arena.latest(initial.lineage_id))

u = execution.diagnostics().state_usage
print("\n结算后用量: 逻辑字节", u.logical_live_bytes, "| 分配载荷", u.allocated_payload_bytes,
      "| 退役页", u.retiring_field_pages, "(被 old_pin 留住) | 被 pin 版本", u.pinned_versions)
d = execution.diagnostics()
print("诊断: update_count =", d.update_count, "| loss_count =", d.loss_count,
      "| trained_rows =", d.trained_rows, "| pending_state_releases =", d.pending_state_releases,
      "| fallback_count =", d.fallback_count)

print("\n本次调用的 trace（注意 PUBLISH 携带 after_refs 与 operation_id）:")
for event in execution.trace.snapshot().events:
    if event.invocation_id == 1 and event.phase in (
            TracePhase.BEGIN, TracePhase.SAMPLER_COMPLETE, TracePhase.PUBLISH, TracePhase.SETTLED):
        print(f"  {event.phase.value:<16} refs={[r.version_id for r in event.refs]} "
              f"op={event.operation_id} pin={event.pin_id} release={event.release_id}")

loss: [1.9196]
发布: 版本 0 -> 版本 1 | operation_id: 2
两个字段都被更新: [True, True]
版本 0 经 old_pin 读出，与训练前逐位一致: True
arena.latest 已指向: StateRef(layout_id=0, lineage_id=1, version_id=1)

结算后用量: 逻辑字节 176 | 分配载荷 176 | 退役页 2 (被 old_pin 留住) | 被 pin 版本 1
诊断: update_count = 1 | loss_count = 3 | trained_rows = 2 | pending_state_releases = 0 | fallback_count = 0

本次调用的 trace（注意 PUBLISH 携带 after_refs 与 operation_id）:
  begin            refs=[0] op=None pin=3 release=None
  sampler_complete refs=[] op=None pin=3 release=None
  publish          refs=[1] op=2 pin=None release=None
  settled          refs=[] op=2 pin=3 release=2


## `state_usage` 的口径，一次说清

结算后的用量里有一处值得停顿：版本 1 已经是最新根，版本 0 的两页却没有归还——它们被 `old_pin` 留住了，记在「退役页」里。这就是精确回收：**页只在引用它的每个 pin 与发布都释放后才归还**。

`PageBundleUsage` 是这篇教程里最值得逐字段认识的结构。它的口径直接写在架构文档「运行时边界」一节和 `usage()` 的 docstring 里：

- 它是从 arena 的根、字段槽、私有写与 receipt **直接推导的冻结快照**，自身不保留这些所有者，也不是另一本账；
- **逻辑字节**把每个保留/私有版本各计一次；**分配载荷**对 COW 共享的字段页只计一次——两者之差就是 COW 省下的物理页；预留填充含容量对齐与余量；
- 空闲、最新根、仅私有、仅退役四类字段页**互斥**（最新根优先于私有，私有优先于退役引用）。它们是所有权类别，不是 GPU 活动度量；
- 被 pin 的版本包括内部写/发布持有与待释放；待收 receipt 包括未收集的**已结算**发布，不只是「还没跑完的 GPU 工作」；
- `admission_capacity_failures` 只覆盖**被拒的准入**，迁移失败或非法输入不计入；
- `arena.usage()` 本身不收集、不同步、不选版本；`execution.diagnostics()` 会先做它常规的非阻塞收集。

回答一个此刻最自然的问题：**COW 什么时候真的拷贝？** 只在迁移发生时，且只为声明可写的字段分配私有页；未写的字段直接共享旧版本的页槽。写多少字段拷多少页——不是整版本复制。

为了把这一点看实，下面绕过训练，直接按 arena 的原语走一遍 `prepare_transition → write_field → publish → settle`：先看全字段迁移（两个私有页），再看只写 `a` 的部分迁移——留意那一刻逻辑字节 264、分配载荷 208，差的 56 字节正是被共享的 `b` 字段页（2×7×4B）。你还会顺路看到：中止（abort）丢弃私有值但保留已发布根；用过期版本发起迁移被响亮拒绝；释放 `old_pin` 之后版本 0 的页立即归还，再读它得到的是明确的错误而不是脏数据。

In [3]:
def show(tag):
    u = arena.usage()  # usage() 本身不收集、不同步、不选版本
    print(f"{tag}: 逻辑={u.logical_live_bytes}B 分配载荷={u.allocated_payload_bytes}B "
          f"空闲页={u.free_field_pages} 最新根={u.live_field_pages} 私有={u.private_field_pages} "
          f"退役={u.retiring_field_pages} 被pin版本={u.pinned_versions} 待收receipt={u.pending_receipts}")

show("步后基线        ")

# 完整迁移：两个字段都可写 -> 两个私有页
token = arena.prepare_transition((latest,))
show("迁移中(全字段)  ")
arena.write_field(token, 0, LayoutFieldId(0)).fill_(1.5)
receipt = arena.publish(token)
show("发布后/结算前   ")
try:
    arena.prepare_transition((latest,))  # 上一版本已不再是 latest
except RuntimeError as exc:
    print("用过期版本发起迁移:", type(exc).__name__, "-", exc)
arena.settle(receipt)
arena.collect_completed()
show("结算并收集后    ")
v2, = receipt.after_refs
print("v2 的 a 全为 1.5:", bool((execution.snapshot(v2)[0] == 1.5).all()))

# 部分字段迁移：只有 a 可写，b 的页被 COW 共享
b_before = execution.snapshot(v2)[1]
token = arena.prepare_transition((v2,), fields=((LayoutFieldId(0),),))
show("迁移中(仅a字段) ")
arena.write_field(token, 0, LayoutFieldId(0)).fill_(2.5)
receipt = arena.publish(token)
arena.settle(receipt)
arena.collect_completed()
v3, = receipt.after_refs
show("部分发布后      ")
print("v3 的 a 全为 2.5:", bool((execution.snapshot(v3)[0] == 2.5).all()),
      "| v3 的 b 与 v2 逐位一致(共享页原样带过来):", bool(torch.equal(execution.snapshot(v3)[1], b_before)))

# 中止：私有值被丢弃，已发布根原样保留
token = arena.prepare_transition((v3,))
arena.write_field(token, 0, LayoutFieldId(0)).fill_(9.9)
arena.abort(token)
show("中止后          ")
print("latest 仍是 v3:", arena.latest(v3.lineage_id) == v3,
      "| v3 的 a 仍是 2.5:", bool((execution.snapshot(v3)[0] == 2.5).all()))

# 释放 old_pin：版本 0 的页立即归还；之后再读它，响亮失败而不是读到脏数据
arena.wait_release(arena.release_pin(old_pin))
show("释放 old_pin 后 ")
try:
    arena.pin_exact((initial,))
except RuntimeError as exc:
    print("再读已回收的版本 0:", type(exc).__name__, "-", exc)
execution.retire(())

步后基线        : 逻辑=176B 分配载荷=176B 空闲页=12 最新根=2 私有=0 退役=2 被pin版本=1 待收receipt=0
迁移中(全字段)  : 逻辑=264B 分配载荷=264B 空闲页=10 最新根=2 私有=2 退役=2 被pin版本=2 待收receipt=0
发布后/结算前   : 逻辑=264B 分配载荷=264B 空闲页=10 最新根=2 私有=0 退役=4 被pin版本=3 待收receipt=1
用过期版本发起迁移: RuntimeError - PageBundle transition source is not latest
结算并收集后    : 逻辑=176B 分配载荷=176B 空闲页=12 最新根=2 私有=0 退役=2 被pin版本=1 待收receipt=0
v2 的 a 全为 1.5: True
迁移中(仅a字段) : 逻辑=264B 分配载荷=208B 空闲页=11 最新根=2 私有=1 退役=2 被pin版本=2 待收receipt=0
部分发布后      : 逻辑=176B 分配载荷=176B 空闲页=12 最新根=2 私有=0 退役=2 被pin版本=1 待收receipt=0
v3 的 a 全为 2.5: True | v3 的 b 与 v2 逐位一致(共享页原样带过来): True
中止后          : 逻辑=176B 分配载荷=176B 空闲页=12 最新根=2 私有=0 退役=2 被pin版本=1 待收receipt=0
latest 仍是 v3: True | v3 的 a 仍是 2.5: True
释放 old_pin 后 : 逻辑=88B 分配载荷=88B 空闲页=14 最新根=2 私有=0 退役=0 被pin版本=0 待收receipt=0
再读已回收的版本 0: RuntimeError - PageBundle root is unknown or reclaimed


(StateRef(layout_id=0, lineage_id=1, version_id=3),)

## 容量与预算：三条 fail-closed 路径

最后把「出事」的情形一次走完。回答三个最常被问到的问题：

**预算超了在哪一步失败？** 规划期，任何分配发生之前。签名计划自带 `peak_memory_bytes`；`memory_budget_bytes` 盖不住它时 `plan_backend` 直接拒绝；负载声明的状态字节超过后端预算时，`specialize` 阶段就拒绝。没有「跑一半才 OOM」。

**pin 没释放会怎样？** 页永远不还——这是设计，不是泄漏。容量宽裕时只是 `state_usage` 里退役页居高不下；容量紧张时，下一次迁移连一页私有页都分不到，于是 fail-closed：`PageBundle capacity is exhausted by live owners`。注意失败语义：`latest` 停在旧版本，没有半成品状态；把欠的 pin 还上，同样的步骤立刻成功。

**准入超上限呢？** 被拒并计入 `admission_capacity_failures`——回忆一下，这个计数只覆盖被拒的准入。请求作用域下每条准入键一条 lineage，所以「准入上限 1」意味着第二个不同键的请求进不来，直到第一个退役。

下面三个小实验分别对应这三条路径。

In [4]:
# (a) 预算在编译期 fail-closed：任何分配发生之前
peak = execution.arena.plan.schedule_decision.peak_memory_bytes
print("签名计划的峰值预算 peak_memory_bytes =", peak)
try:
    workload.plan_backend(workload.specialize(base_facts, ExecutionOptions(training_rows=4, memory_budget_bytes=0)))
except ValueError as exc:
    print("memory_budget_bytes=0 -> 规划期拒绝:", str(exc)[:60], "...")
try:
    workload.specialize(replace(base_facts, state_memory_budget_bytes=16384),
        ExecutionOptions(training_rows=4, state_memory_bytes=32768))
except ValueError as exc:
    print("状态请求超过后端预算 -> 特化期拒绝:", exc)

# (b) pin 没释放会怎样：容量为 2 的 arena 里，持住的 pin 让下一步无处可写
tight_workload = OutputLoRAWorkload(config)
tight = tight_workload.compile_backend(tight_workload.specialize(
    replace(base_facts, providers=(PageBundleProviderFacts("test.page@1", 2),)),
    ExecutionOptions(training_rows=4))).execution
t_initial, = tight.admit(())
t_pin = tight.arena.pin_exact((t_initial,))
one_rows = PackedPhaseRowBatch(1, 0, 1, (PackedPhaseRow(0, 0, 0, HiddenPhase.DECODE, RequestIdentity("0", 0, 0)),))
one_row = TensorStepInput(one_rows, PackedSamplerRowBatch(one_rows, (PackedSamplerRowLink(0, 0),)),
    PackedHiddenSnapshotBatch(one_rows, (PackedHiddenSnapshot(binding, hidden3[:1]),)), auxiliary=batch)
step1 = tight.step(one_row)
print("\n紧凑 arena: 第一步发布到版本", step1.publication.after_refs[0].version_id)
try:
    tight.step(one_row)
except RuntimeError as exc:
    print("第二步被拒(fail-closed):", exc)
print("latest 仍停在版本", tight.arena.latest(t_initial.lineage_id).version_id, "——没有半成品状态")
tight.arena.wait_release(tight.arena.release_pin(t_pin))
step2 = tight.step(one_row)
print("释放 pin 后第二步成功: 版本", step2.publication.after_refs[0].version_id)
tight.retire(())

# (c) 准入容量：被拒的准入计入 admission_capacity_failures
request_scoped = OutputLoRAWorkload(replace(config, scope=StateScope.REQUEST))
r_exec = request_scoped.compile_backend(request_scoped.specialize(
    replace(base_facts, max_admitted_lineages=1), ExecutionOptions(training_rows=4))).execution
ref_a, = r_exec.admit((RequestIdentity("a", 0, 0),))
print("\n请求 a 准入:", ref_a)
try:
    r_exec.admit((RequestIdentity("b", 1, 0),))
except RuntimeError as exc:
    print("请求 b 准入被拒:", exc)
print("admission_capacity_failures =", r_exec.diagnostics().state_usage.admission_capacity_failures)
r_exec.retire((RequestIdentity("a", 0, 0),))
ref_b, = r_exec.admit((RequestIdentity("b", 1, 0),))
print("退役 a 后 b 准入成功:", ref_b)
r_exec.retire((RequestIdentity("b", 1, 0),))

签名计划的峰值预算 peak_memory_bytes = 3080
memory_budget_bytes=0 -> 规划期拒绝: no joint state representation and AER placement fits version ...
状态请求超过后端预算 -> 特化期拒绝: requested state capacity exceeds available backend memory

紧凑 arena: 第一步发布到版本 1
第二步被拒(fail-closed): PageBundle capacity is exhausted by live owners
latest 仍停在版本 1 ——没有半成品状态
释放 pin 后第二步成功: 版本 2

请求 a 准入: StateRef(layout_id=0, lineage_id=1, version_id=0)
请求 b 准入被拒: PageBundle admission capacity is exhausted
admission_capacity_failures = 1
退役 a 后 b 准入成功: StateRef(layout_id=0, lineage_id=2, version_id=0)


(StateRef(layout_id=0, lineage_id=2, version_id=0),)

## 小结与下一步

回头看，这篇把签名计划落地后的所有权纪律走完了：

- **arena 是唯一所有者**：版本、代次、完成都归 `PageBundleArena`；执行只读 pin 住的版本，写入私有页，发布原子换根；
- **COW 是字段级的**：未写字段共享页槽，`state_usage` 里逻辑字节与分配载荷之差就是它省下的物理页；
- **回收由 pin 与 receipt 门控**：旧版本活到最后一个 pin 释放、最后一张 receipt 结清；之后再读它是响亮失败，不是脏数据；
- **容量与预算都 fail-closed**：预算在规划期拒绝，准入在运行期拒绝并计数，持住的 pin 会堵住后续迁移，但绝不产生半成品状态。

如果只能带走一句话的心智模型，就这句：**arena 里只有一种真相——被 pin 的不可变版本；写是借私有页起一个新版本，发布是原子换根，回收等所有 pin 与 receipt 结清。**

支线第 3 篇我们将跨出进程边界：这些页如何按引用（而不是拷贝）交给另一个进程，worker 又如何只校验固定见证、而不重跑任何优化。想回主线看请求作用域状态在真实引擎里的样子，去 [教程 3：按请求路由的 LoRA](../03_request_lora.ipynb)；想读实现，源码在 `pllm/execution/ptts/page_bundle.py` 与 `pllm/execution/state_execution.py`，口径原文在 `doc/design/architecture.md` 的「运行时边界」一节。